<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0 Fixed Grid Backtest

Single source of truth: configuration, fixed-grid engine, audit, and GitHub log export all live in this notebook.

**Execution assumptions**
- BUY only on downward crossing of a grid level.
- Existing SELL targets are processed before new BUYs in each 1-minute candle.
- A new BUY cannot SELL in the same candle.
- A level sold in the current candle cannot rebuy in that candle.
- Same-candle SELL proceeds are not reused for BUYs.


## 1. Setup & User Configuration


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import bisect
import heapq
import json
import os

import numpy as np
import pandas as pd
import requests

# Market data
DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"
SYMBOL = "BTCUSDT"
TIMEFRAME = "1m"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"

# Strategy
INITIAL_CAPITAL = 3000.0
GRID_FLOOR = 38000.0
GRID_CEILING = 127000.0
GRID_GAP = 1000.0
BUY_FEE = 0.001
SELL_FEE = 0.001

# Log output
REPO = "natdanaiii/Trading"
BRANCH = "main"
GITHUB_LOG_PATH = "logs/latest_v0_backtest_log.json"
LOCAL_LOG_PATH = "/content/latest_v0_backtest_log.json"


## 2. Load Market Data


In [ ]:
def load_market_data(symbol, timeframe, data_dir):
    """Load, validate, sort, and filter the historical OHLCV data."""

    file_path = os.path.join(
        data_dir,
        f"{symbol}-{timeframe}-combined.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    data = pd.read_csv(file_path)

    required_columns = {
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }
    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns: {sorted(missing_columns)}"
        )

    data["open_time"] = pd.to_datetime(
        data["open_time"],
        utc=True,
    )

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]
    data[numeric_columns] = data[numeric_columns].astype(float)

    start_time = pd.Timestamp(START_DATE, tz="UTC")
    end_time = pd.Timestamp(END_DATE, tz="UTC")

    data = (
        data
        .drop_duplicates("open_time")
        .sort_values("open_time")
        .loc[
            lambda df:
            (df["open_time"] >= start_time)
            & (df["open_time"] < end_time)
        ]
        .reset_index(drop=True)
    )

    if data.empty:
        raise ValueError("No market data inside the selected period.")

    return data


## 3. Build Fixed Grid


In [ ]:
def build_fixed_grid_table(
    capital,
    floor,
    ceiling,
    gap,
    buy_fee,
    sell_fee,
):
    """Build the arithmetic grid and pre-calculate each BUY/SELL cycle."""

    if capital <= 0:
        raise ValueError("capital must be greater than 0.")
    if floor <= 0:
        raise ValueError("floor must be greater than 0.")
    if ceiling <= floor:
        raise ValueError("ceiling must be greater than floor.")
    if gap <= 0:
        raise ValueError("gap must be greater than 0.")
    if not (0 <= buy_fee < 1):
        raise ValueError("buy_fee must be in [0, 1).")
    if not (0 <= sell_fee < 1):
        raise ValueError("sell_fee must be in [0, 1).")

    raw_grid_count = (ceiling - floor) / gap

    if not np.isclose(raw_grid_count, round(raw_grid_count)):
        raise ValueError(
            "(ceiling - floor) must be exactly divisible by gap."
        )

    number_of_grids = int(round(raw_grid_count))
    capital_per_grid = capital / number_of_grids

    buy_prices = (
        ceiling
        - gap * np.arange(1, number_of_grids + 1)
    )
    sell_prices = buy_prices + gap

    gross_btc = capital_per_grid / buy_prices
    buy_fee_btc = gross_btc * buy_fee
    net_btc = gross_btc - buy_fee_btc

    gross_sell_usdt = net_btc * sell_prices
    sell_fee_usdt = gross_sell_usdt * sell_fee
    net_sell_usdt = gross_sell_usdt - sell_fee_usdt

    cycle_profit = net_sell_usdt - capital_per_grid

    return pd.DataFrame(
        {
            "level": np.arange(1, number_of_grids + 1),
            "buy_price": buy_prices,
            "sell_price": sell_prices,
            "capital_per_grid": capital_per_grid,
            "base_amount": net_btc,
            "buy_fee_base": buy_fee_btc,
            "sell_fee_quote": sell_fee_usdt,
            "net_sell": net_sell_usdt,
            "profit": cycle_profit,
        }
    )


## 4. Performance Statistics


In [ ]:
def performance_stats(data, equity, initial_capital):
    """Calculate return, drawdown, annualized return, and Calmar ratio."""

    running_peak = np.maximum.accumulate(equity)
    drawdown = equity / running_peak - 1.0

    final_equity = float(equity[-1])
    max_drawdown = float(drawdown.min())
    net_return = final_equity / initial_capital - 1.0

    elapsed_days = (
        data["open_time"].iloc[-1]
        - data["open_time"].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan

    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = (
            np.log(final_equity / initial_capital)
            * (365.25 / elapsed_days)
        )

        if annualized_log_growth < 700:
            annualized_return = float(
                np.expm1(annualized_log_growth)
            )

    calmar_ratio = np.nan

    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar_ratio = float(
            annualized_return / abs(max_drawdown)
        )

    return (
        final_equity,
        net_return,
        annualized_return,
        max_drawdown,
        calmar_ratio,
        drawdown,
    )


## 5. V0 Backtest Engine

For each 1-minute candle:

1. Process existing SELL targets.
2. Process grid levels crossed downward for new BUYs.
3. Mark portfolio equity at the candle Close.


In [ ]:
def run_fixed_grid_backtest(
    df_price,
    grid,
    initial_capital,
):
    """Run the fixed-grid strategy over historical 1-minute candles."""

    data = (
        df_price
        .sort_values("open_time")
        .reset_index(drop=True)
    )
    grid = (
        grid
        .sort_values("buy_price")
        .reset_index(drop=True)
        .copy()
    )

    if data.empty:
        raise ValueError("df_price is empty.")
    if grid.empty:
        raise ValueError("grid is empty.")

    # Static grid arrays
    buy_prices = grid["buy_price"].to_numpy(float)
    sell_prices = grid["sell_price"].to_numpy(float)
    order_costs = grid["capital_per_grid"].to_numpy(float)
    btc_amounts = grid["base_amount"].to_numpy(float)
    buy_fee_btc = grid["buy_fee_base"].to_numpy(float)
    sell_fee_usdt = grid["sell_fee_quote"].to_numpy(float)
    net_sell_usdt = grid["net_sell"].to_numpy(float)
    cycle_profits = grid["profit"].to_numpy(float)
    buy_price_list = buy_prices.tolist()

    # Position state
    holding = np.zeros(len(grid), dtype=bool)
    buy_times = [None] * len(grid)
    sell_heap = []

    # Portfolio state
    cash = float(initial_capital)
    btc = 0.0
    realized_profit = 0.0
    total_buy_fee_usdt = 0.0
    total_sell_fee_usdt = 0.0
    completed_cycles = 0

    # Logs
    trade_events = []
    completed_trades = []
    event_id = 0

    # Time-series outputs
    row_count = len(data)
    equity_values = np.empty(row_count)
    cash_values = np.empty(row_count)
    btc_values = np.empty(row_count)

    previous_close = None

    for row_index, candle in enumerate(data.itertuples(index=False)):
        timestamp = candle.open_time
        open_price = float(candle.open)
        high_price = float(candle.high)
        low_price = float(candle.low)
        close_price = float(candle.close)

        # New BUYs may only use cash available at candle start.
        cash_at_candle_start = cash
        sold_this_candle = set()

        # ----------------------------------------------------
        # 1) Process existing SELL targets
        # ----------------------------------------------------
        while sell_heap and sell_heap[0][0] <= high_price:
            _, grid_index = heapq.heappop(sell_heap)

            if not holding[grid_index]:
                continue

            cash_before = cash
            btc_before = btc

            holding[grid_index] = False
            cash += net_sell_usdt[grid_index]
            btc -= btc_amounts[grid_index]

            if abs(btc) < 1e-12:
                btc = 0.0

            realized_profit += cycle_profits[grid_index]
            total_sell_fee_usdt += sell_fee_usdt[grid_index]
            completed_cycles += 1
            sold_this_candle.add(grid_index)
            event_id += 1

            completed_trades.append(
                {
                    "buy_time": buy_times[grid_index],
                    "sell_time": timestamp,
                    "buy_price": buy_prices[grid_index],
                    "sell_price": sell_prices[grid_index],
                    "cost": order_costs[grid_index],
                    "profit": cycle_profits[grid_index],
                }
            )

            trade_events.append(
                {
                    "event_id": event_id,
                    "time": timestamp,
                    "side": "SELL",
                    "price": sell_prices[grid_index],
                    "cash_movement": net_sell_usdt[grid_index],
                    "grid_cashflow": cycle_profits[grid_index],
                    "cash_before": cash_before,
                    "cash_after": cash,
                    "btc_before": btc_before,
                    "btc_after": btc,
                }
            )

            buy_times[grid_index] = None

        # ----------------------------------------------------
        # 2) Process downward BUY crossings
        # ----------------------------------------------------
        buy_budget = cash_at_candle_start

        downward_start = (
            open_price
            if previous_close is None
            else max(previous_close, open_price)
        )

        if low_price < downward_start:
            first_index = bisect.bisect_left(
                buy_price_list,
                low_price,
            )
            stop_index = bisect.bisect_left(
                buy_price_list,
                downward_start,
            )

            # Higher crossed levels are reached first as price falls.
            for grid_index in range(
                stop_index - 1,
                first_index - 1,
                -1,
            ):
                if holding[grid_index]:
                    continue
                if grid_index in sold_this_candle:
                    continue

                order_cost = order_costs[grid_index]

                if buy_budget + 1e-12 < order_cost:
                    break

                cash_before = cash
                btc_before = btc

                holding[grid_index] = True
                buy_times[grid_index] = timestamp

                buy_budget -= order_cost
                cash -= order_cost
                btc += btc_amounts[grid_index]

                total_buy_fee_usdt += (
                    buy_fee_btc[grid_index]
                    * buy_prices[grid_index]
                )

                heapq.heappush(
                    sell_heap,
                    (sell_prices[grid_index], grid_index),
                )

                event_id += 1
                trade_events.append(
                    {
                        "event_id": event_id,
                        "time": timestamp,
                        "side": "BUY",
                        "price": buy_prices[grid_index],
                        "cash_movement": -order_cost,
                        "grid_cashflow": 0.0,
                        "cash_before": cash_before,
                        "cash_after": cash,
                        "btc_before": btc_before,
                        "btc_after": btc,
                    }
                )

        # ----------------------------------------------------
        # 3) Mark portfolio at candle Close
        # ----------------------------------------------------
        equity_values[row_index] = cash + btc * close_price
        cash_values[row_index] = cash
        btc_values[row_index] = btc
        previous_close = close_price

    (
        final_equity,
        net_return,
        annualized_return,
        max_drawdown,
        calmar_ratio,
        drawdown,
    ) = performance_stats(
        data,
        equity_values,
        initial_capital,
    )

    equity_curve = pd.DataFrame(
        {
            "open_time": data["open_time"],
            "close": data["close"],
            "cash": cash_values,
            "btc": btc_values,
            "equity": equity_values,
            "drawdown": drawdown,
        }
    )
    trade_log = pd.DataFrame(trade_events)

    summary = {
        "initial_capital": initial_capital,
        "final_equity": final_equity,
        "net_return": net_return,
        "annualized_return": annualized_return,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "completed_cycles": completed_cycles,
        "open_positions": int(holding.sum()),
        "final_cash": cash,
        "final_btc": btc,
        "realized_profit": realized_profit,
        "unrealized_pnl": final_equity - initial_capital - realized_profit,
        "total_fee_usdt_equiv": total_buy_fee_usdt + total_sell_fee_usdt,
    }

    return {
        "summary": summary,
        "trade_log": trade_log,
        "completed_trades": pd.DataFrame(completed_trades),
        "equity_curve": equity_curve,
        "grid_state": grid.assign(holding=holding, buy_time=buy_times),
    }


## 6. System Audit


In [ ]:
def audit_v0(result):
    """Check accounting identities and basic portfolio safety."""

    summary = result["summary"]
    trade_log = result["trade_log"]
    equity_curve = result["equity_curve"]

    if trade_log.empty:
        cash_movement = 0.0
        grid_cashflow = 0.0
    else:
        cash_movement = trade_log["cash_movement"].sum()
        grid_cashflow = trade_log["grid_cashflow"].sum()

    cash_reconciliation_error = abs(
        INITIAL_CAPITAL
        + cash_movement
        - summary["final_cash"]
    )
    realized_profit_error = abs(
        grid_cashflow
        - summary["realized_profit"]
    )
    equity_identity_error = float(
        np.max(
            np.abs(
                equity_curve["cash"]
                + equity_curve["btc"] * equity_curve["close"]
                - equity_curve["equity"]
            )
        )
    )
    minimum_cash = float(equity_curve["cash"].min())

    return {
        "cash_reconciliation": cash_reconciliation_error <= 1e-8,
        "realized_profit_reconciliation": realized_profit_error <= 1e-8,
        "equity_identity": equity_identity_error <= 1e-8,
        "cash_never_negative": minimum_cash >= -1e-8,
    }


## 7. Run Backtest & Review Results


In [ ]:
df_1m = load_market_data(
    SYMBOL,
    TIMEFRAME,
    DATA_DIR,
)

grid = build_fixed_grid_table(
    capital=INITIAL_CAPITAL,
    floor=GRID_FLOOR,
    ceiling=GRID_CEILING,
    gap=GRID_GAP,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
)

result = run_fixed_grid_backtest(
    df_price=df_1m,
    grid=grid,
    initial_capital=INITIAL_CAPITAL,
)

summary = result["summary"]
audit_checks = audit_v0(result)
AUDIT_STATUS = "PASS" if all(audit_checks.values()) else "FAIL"

print("===== DATA =====")
print(f"Rows            : {len(df_1m):,}")
print(
    "Period          : "
    f"{df_1m['open_time'].min()} "
    f"-> {df_1m['open_time'].max()}"
)

print("\n===== V0 CONFIGURATION =====")
print(f"Initial Capital : {INITIAL_CAPITAL:,.2f} USDT")
print(f"Floor           : {GRID_FLOOR:,.2f} USDT")
print(f"Ceiling         : {GRID_CEILING:,.2f} USDT")
print(f"Gap             : {GRID_GAP:,.2f} USDT")
print(f"Number of Grids : {len(grid)}")
print(
    "Capital / Grid  : "
    f"{grid['capital_per_grid'].iloc[0]:,.6f} USDT"
)

print("\n===== V0 RESULT =====")
print(f"Final Equity    : {summary['final_equity']:,.2f} USDT")
print(f"Net Return      : {summary['net_return']:.2%}")
print(f"Annualized      : {summary['annualized_return']:.2%}")
print(f"Max Drawdown    : {summary['max_drawdown']:.2%}")
print(f"Calmar Ratio    : {summary['calmar_ratio']:.3f}")
print(f"Completed Cycles: {summary['completed_cycles']:,}")
print(f"Open Positions  : {summary['open_positions']:,}")
print(f"Final Cash      : {summary['final_cash']:,.2f} USDT")
print(f"Final BTC       : {summary['final_btc']:.8f} BTC")
print(f"Realized Profit : {summary['realized_profit']:,.2f} USDT")
print(f"Unrealized P&L  : {summary['unrealized_pnl']:,.2f} USDT")
print(f"Total Fees      : {summary['total_fee_usdt_equiv']:,.2f} USDT")

print("\n===== V0 AUDIT =====")
for check_name, passed in audit_checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{check_name:32s}: {status}")

print(f"{'Overall':32s}: {AUDIT_STATUS}")

if AUDIT_STATUS != "PASS":
    raise AssertionError("V0 AUDIT FAILED")


## 8. Export Latest Log to GitHub


In [ ]:
def json_safe(value):
    """Convert NumPy/Pandas values into JSON-safe Python values."""

    if value is pd.NaT or value is pd.NA:
        return None
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return [json_safe(item) for item in value.tolist()]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def build_log_payload(market_data, grid_table, summary, audit_checks, audit_status):
    """Build the compact JSON log used for later review."""

    return {
        "log_schema_version": 2,
        "strategy": "V0 Fixed Grid",
        "run_info": {
            "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "repository": REPO,
            "branch": BRANCH,
            "notebook": "Grid_trading_V0.ipynb",
            "symbol": SYMBOL,
            "timeframe": TIMEFRAME,
            "start_date": START_DATE,
            "end_date": END_DATE,
            "data_rows": int(len(market_data)),
            "data_first_time": market_data["open_time"].min().isoformat(),
            "data_last_time": market_data["open_time"].max().isoformat(),
        },
        "parameters": {
            "initial_capital": INITIAL_CAPITAL,
            "floor": GRID_FLOOR,
            "ceiling": GRID_CEILING,
            "gap": GRID_GAP,
            "buy_fee": BUY_FEE,
            "sell_fee": SELL_FEE,
        },
        "derived": {
            "number_of_grids": int(len(grid_table)),
            "capital_per_grid": float(grid_table["capital_per_grid"].iloc[0]),
        },
        "market_range_diagnostics": {
            "historical_low": float(market_data["low"].min()),
            "historical_high": float(market_data["high"].max()),
            "candles_low_below_floor": int((market_data["low"] < GRID_FLOOR).sum()),
            "candles_high_above_ceiling": int((market_data["high"] > GRID_CEILING).sum()),
        },
        "summary": summary,
        "audit": {
            "status": audit_status,
            "checks": audit_checks,
        },
    }


def upload_log_to_github(log_payload):
    """Upload the latest log if the Colab GITHUB_TOKEN exists."""

    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if not github_token:
        print("GitHub upload SKIPPED: Colab Secret 'GITHUB_TOKEN' was not found.")
        return

    api_url = f"https://api.github.com/repos/{REPO}/contents/{GITHUB_LOG_PATH}"
    headers = {
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    existing_file = requests.get(api_url, headers=headers, timeout=30)

    body = {
        "message": "Update latest V0 backtest log",
        "content": base64.b64encode(
            json.dumps(log_payload, indent=2).encode("utf-8")
        ).decode("utf-8"),
        "branch": BRANCH,
    }

    if existing_file.status_code == 200:
        body["sha"] = existing_file.json()["sha"]

    upload = requests.put(
        api_url,
        headers=headers,
        json=body,
        timeout=30,
    )
    upload.raise_for_status()

    print("GitHub log upload: SUCCESS")
    print(f"Path      : {GITHUB_LOG_PATH}")
    print(f"Commit SHA: {upload.json()['commit']['sha']}")


log_payload = build_log_payload(
    market_data=df_1m,
    grid_table=grid,
    summary=summary,
    audit_checks=audit_checks,
    audit_status=AUDIT_STATUS,
)
log_payload = json_safe(log_payload)

with open(LOCAL_LOG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        log_payload,
        file,
        indent=2,
        allow_nan=False,
    )

print(f"Local log : {LOCAL_LOG_PATH}")
upload_log_to_github(log_payload)
